# Automação de Extração do Relatório de Atestados Médicos

- Extrair do sistema Datamace o relatório de afastamentos através do módulo Mediciona Ocupacional
- Tratamento da base ATESTADOS, padronização de nomes de colunas e formatos de dados, além da atualização da planilha Controle HC e Atestados

# Importando as Bibliotecas

In [11]:
import logging
import os
import pandas as pd
import pyautogui
import pyperclip
import shutil
import time
import tkinter as tk
import win32com.client
from asyncio.log import logger
from contextlib import contextmanager
from datetime import datetime, date
from openpyxl import load_workbook, Workbook
from openpyxl.utils.dataframe import dataframe_to_rows
from openpyxl.worksheet.table import Table, TableStyleInfo
from pathlib import Path
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver import ActionChains
from tkinter import simpledialog
from webdriver_manager.chrome import ChromeDriverManager

tempo_0 = [id, datetime.today(), 0]

print('----------------------------------------------------------------------------------------------------')
print('')
print('   ✅ Processo de Importação finalizado')
print('')
print('----------------------------------------------------------------------------------------------------')

----------------------------------------------------------------------------------------------------

   ✅ Processo de Importação finalizado

----------------------------------------------------------------------------------------------------


# Caminhos de Pastas

In [12]:
# Configuração
CONFIG = {
    'id_processo': 4,
    'processos': Path(r'X:\Gestao_de_Pessoas\Analytics\03 - Bases\1. BASES TRATADAS\PROCESSOS.xlsx'),
    'origem': Path(r'C:\Users\rodrigo.bernandes\Downloads'),
    'movidos': Path(r'X:\Gestao_de_Pessoas\Analytics\03 - Bases\2. ARQUIVOS MOVIDOS'),
    'saida': Path(r'X:\Gestao_de_Pessoas\Analytics\03 - Bases\1. BASES TRATADAS\ATESTADOS.xlsx'),
    'padrao': 'RFA13C',
}

# Registra etapa do processo

In [4]:
def append_registro_processo(caminho, id_proc, etapa):
    try:
        wb = load_workbook(caminho)
        ws = wb['REGISTROS']
        ws.append([id_proc, datetime.today(), etapa])
        wb.save(caminho)
        wb.close()
    except Exception as e:
        print(f"❌ Erro ao registrar etapa {etapa}: {e}")

print('----------------------------------------------------------------------------------------------------')
print('')
print('   ✅ Registro de processos')
print('')
print('----------------------------------------------------------------------------------------------------')

----------------------------------------------------------------------------------------------------

   ✅ Registro de processos

----------------------------------------------------------------------------------------------------


# Configurações Iniciais

In [4]:
automacao = r'X:\Gestao_de_Pessoas\Analytics\08 - Notebooks Python\08.3 - Automações\AUTOMACAO'

chrome_options = Options()
chrome_options.add_argument(f"--user-data-dir={automacao}")
chrome_options.add_argument("--profile-directory=Default")
chrome_options.add_argument("--remote-allow-origins=*")
chrome_options.add_argument("--start-maximized")
service = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service, options=chrome_options)

try:
    driver.get("https://app.dtm.com.br/")
    wait = WebDriverWait(driver, 150)
    actions = ActionChains(driver)
    
    print("⏳ Sincronizando Datamace...")

except Exception as e:
    print(f"❌ Erro Geral: {e}")

finally:
    print(f"🏁 Acesso finalizado")

⏳ Sincronizando Datamace...
🏁 Acesso finalizado


# Acessando o Datamace

In [5]:
pyautogui.PAUSE = 1

# Área de login na Cloud Datamace
pyautogui.click(x=748, y=381, clicks=2)
pyautogui.write("afpesp-17")
pyautogui.press("tab")
pyautogui.write("5SSWi3VqS")
pyautogui.press("tab")
pyautogui.press("enter")
time.sleep(30) # Tempo maior para aguardar carregar a página de login

# Área de login do usuário
pyautogui.click(x=836, y=445)
pyautogui.write("RODRIGO")
pyautogui.press("tab")
pyautogui.write("Roddydio/")
pyautogui.press("tab")
pyautogui.press("enter")

print('----------------------------------------------------------------------------------------------------')
print('')
print('   ✅ Acesso ao Datamace realizado com sucesso')
print('')
print('----------------------------------------------------------------------------------------------------')

----------------------------------------------------------------------------------------------------

   ✅ Acesso ao Datamace realizado com sucesso

----------------------------------------------------------------------------------------------------


# Dentro do Datamace, acessando o MO

In [5]:
# Acessando o MO
time.sleep(10) # Tempo maior para aguardar carregar o MO
pyautogui.click(x=474, y=196) # Clicando no módulo MO
time.sleep(3)
pyautogui.click(x=1079, y=235) # Fecha a janela de Tarefas Anuais
time.sleep(2)
pyautogui.click(x=52, y=299) # Ambulatório
time.sleep(1)
pyautogui.click(x=147, y=590) # Atestado Médico
time.sleep(1)
pyautogui.click(x=313, y=627) # Relatório Atestado Médico
time.sleep(1)
pyautogui.click(x=531, y=285) # Multi processamento
time.sleep(1)
pyautogui.click(x=862, y=599) # Situação
time.sleep(1)
pyautogui.click(x=850, y=642) # Todos
time.sleep(1)
pyautogui.click(x=776, y=645) # Observação
time.sleep(1)
pyautogui.click(x=675, y=651) # Saída
time.sleep(1)
pyautogui.click(x=659, y=682) # Excel
time.sleep(1)
pyautogui.click(x=557, y=717) # Confirmar

print('----------------------------------------------------------------------------------------------------')
print('')
print('   ✅ Extraindo relatório')
print('')
print('----------------------------------------------------------------------------------------------------')

----------------------------------------------------------------------------------------------------

   ✅ Extraindo relatório

----------------------------------------------------------------------------------------------------


# Aguardando a Conclusão do Download da base ATESTADOS

In [7]:
root = tk.Tk()
root.withdraw()  # não mostra janela principal

while True:
    tecla = simpledialog.askstring("Continuar", "Digite S para continuar:")
    if (tecla or "").strip().upper() == "S":
        break

print(" ✅ Continuando o processo...")

 ✅ Continuando o processo...


# Mapeamento de colunas

In [13]:
MAPEAMENTO_COLUNAS = {
    'Cod.Emp.': 'cod_empresa',
    'Registro': 'registro',
    'Nome Trab.': 'nome',
    'CPF': 'cpf',
    'Data Nasc.': 'nascimento',
    'Sexo': 'sexo',
    'Data': 'data_atestado',
    'Tab.Cons.': 'cod_consultorio',
    'Nome Cons.': 'consultorio',
    'Tab.Med.': 'cod_medico',
    'Nome Med.': 'medico',
    'Conselho': 'conselho',
    'Num.Cons.': 'registro_conselho',
    'Data Ini.': 'data_inicio',
    'Data Final': 'data_fim',
    'Tempo': 'tempo',
    'Tipo Tem.': 'tipo_tempo',
    'Hor. entr.': 'hora_entrada',
    'Hor. saída': 'hora_saida',
    'CID': 'cid',
    'Desc.CID': 'descricao_cid',
    'Capitulo': 'capitulo',
    'Grupo Ini.': 'grupo_inicial',
    'Grupo Fin.': 'grupo_final',
    'Categoria': 'categoria',
    'NTEP': 'ntep',
    'Aceito': 'aceito',
    'Motivo': 'motivo',
    'Observacao': 'observacao',
    'Des.Cargo': 'cargo_abreviado',
    'Data inc.': 'data_inclusao',
    'Usu.incl.': 'usuario_inclusao',
    'Data alt.': 'data_alteracao',
    'Usu.alte.': 'usuario_alteracao',
    'Hr.não.tr.': 'horas_nao_trabalhadas',
}
COLUNAS_DATA = ['Data Nasc.', 'Data', 'Data Ini.', 'Data Final', 'Data inc.', 'Data alt.']

# Processando Atestados Médicos

In [14]:
@contextmanager

def gerenciar_workbook(caminho: Path, sheet: str):
    """Context manager para operações seguras com workbook."""
    wb = load_workbook(caminho)
    ws = wb[sheet]
    try:
        yield ws
    finally:
        wb.save(caminho)

def registrar_etapa(caminho: Path, id_proc: int, etapa: int):
    """Registra etapa do processo."""
    with gerenciar_workbook(caminho, 'REGISTROS') as ws:
        ws.append([id_proc, datetime.today(), etapa])
    logger.debug(f"✅ Etapa {etapa} registrada")

def converter_datas(df: pd.DataFrame, colunas: list) -> pd.DataFrame:
    """Converte colunas para datetime."""
    for col in colunas:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], format='%d/%m/%Y', errors='coerce')
    return df

def limpar_registro(df: pd.DataFrame) -> pd.DataFrame:
    """Limpa coluna REGISTRO (pega apenas primeiros 6 caracteres)."""
    if 'Registro' in df.columns:
        df['Registro'] = df['Registro'].astype(str).str[:6].str.strip()
    return df

def limpar_tempo(df: pd.DataFrame) -> pd.DataFrame:
    """Converte campo Tempo de string para número."""
    if 'Tempo' in df.columns:
        df['Tempo'] = pd.to_numeric(
            df['Tempo'].astype(str).str.replace(',', '.'),
            errors='coerce'
        )
    return df

def mapear_colunas_atestados(df: pd.DataFrame) -> pd.DataFrame:
    """Mapeia colunas do arquivo de atestados."""
    colunas_existentes = {k: v for k, v in MAPEAMENTO_COLUNAS.items() if k in df.columns}
    return df.rename(columns=colunas_existentes)[list(colunas_existentes.values())]

def limpar_valores_nulos(df: pd.DataFrame) -> pd.DataFrame:
    """Limpa valores NaN e converte para None (compatível com Excel)."""
    # Python 3.10+: usar .map() ao invés de .applymap()
    try:
        return df.map(lambda x: None if pd.isna(x) else x)
    except AttributeError:
        # Fallback para versões mais antigas
        return df.applymap(lambda x: None if pd.isna(x) else x)
    
def processar_arquivo(caminho: Path) -> pd.DataFrame:
    """Processa um arquivo de atestados."""
    try:
        logger.debug(f"Carregando: {caminho.name}")
        
        # Carregar
        df = pd.read_excel(caminho, engine='openpyxl')
        
        # Converter tipos
        df = converter_datas(df, COLUNAS_DATA)
        df = limpar_registro(df)
        df = limpar_tempo(df)
        
        # Mapear colunas
        df = mapear_colunas_atestados(df)
        
        # Limpar NaNs
        df = limpar_valores_nulos(df)
        
        logger.debug(f"✅ {len(df)} registros processados")
        return df
        
    except Exception as e:
        logger.error(f"❌ Erro ao processar {caminho.name}: {e}")
        import traceback
        logger.error(traceback.format_exc())
        return None
    
def criar_excel_com_tabela(df: pd.DataFrame, caminho: Path):
    """Cria Excel com tabela formatada."""
    logger.info(f"📝 Criando Excel final ({len(df)} registros)...")
    
    # Salvar com Pandas
    df.to_excel(caminho, sheet_name='ATESTADOS', index=False, engine='openpyxl')
    
    # Aplicar formatação
    wb = load_workbook(caminho)
    ws = wb.active
    
    tabela = Table(displayName="ATESTADOS", ref=ws.dimensions)
    estilo = TableStyleInfo(
        name="TableStyleLight13",
        showFirstColumn=False,
        showLastColumn=False,
        showRowStripes=True,
        showColumnStripes=True
    )
    tabela.tableStyleInfo = estilo
    ws.add_table(tabela)
    
    wb.save(caminho)
    logger.info(f"✅ Excel criado: {caminho}")

# Main
if __name__ == "__main__":
    tempo_inicio = datetime.now()
    
    logger.info("=" * 80)
    logger.info("🔄 PROCESSAMENTO DE ATESTADOS")
    logger.info("=" * 80)

# Processamento e Consolidação do Arquivo

In [15]:
try:
        # Etapa 1: Registrar início
        registrar_etapa(CONFIG['processos'], CONFIG['id_processo'], 0)
            
        # Etapa 2: Buscar arquivos
        logger.info("\n📂 Buscando arquivos...")
        arquivos = sorted([f for f in CONFIG['origem'].iterdir() if f.name.startswith(CONFIG['padrao'])])
            
        if not arquivos:
            logger.warning("❌ Nenhum arquivo encontrado")
            exit()
            
        logger.info(f"✅ {len(arquivos)} arquivo(s) encontrado(s)\n")
            
        registrar_etapa(CONFIG['processos'], CONFIG['id_processo'], 1)
            
        # Etapa 3: Processar arquivos (CONSOLIDAR FORA DO LOOP)
        logger.info("🔄 Processando arquivos...\n")
            
        todas_bases = []
            
        for idx, arquivo in enumerate(arquivos, 1):
            logger.info(f"[{idx}/{len(arquivos)}] {arquivo.name}")
                
            df = processar_arquivo(arquivo)
                
            if df is not None and len(df) > 0:
                todas_bases.append(df)
                logger.info(f"   ✅ {len(df)} registros adicionados")
                    
                try:
                    destino = CONFIG['movidos'] / arquivo.name
                    shutil.move(str(arquivo), str(destino))
                    logger.info(f"   ✅ Movido para arquivos processados\n")
                except Exception as e:
                    logger.error(f"   ⚠️ Erro ao mover: {e}\n")
            else:
                logger.warning(f"   ❌ Sem dados válidos\n")
            
        registrar_etapa(CONFIG['processos'], CONFIG['id_processo'], 2)
            
        # Etapa 4: Consolidar e salvar (UMA ÚNICA VEZ)
        if todas_bases:
            logger.info("💾 Consolidando dados...")
            base_final = pd.concat(todas_bases, ignore_index=True)
            base_final = base_final.drop_duplicates()
            logger.info(f"✅ {len(base_final)} registros consolidados\n")
                
            criar_excel_com_tabela(base_final, CONFIG['saida'])
        else:
            logger.error("❌ Nenhum arquivo foi processado!")
            exit()
            
        # Finalizar
        tempo_duracao = datetime.now() - tempo_inicio
            
        logger.info("\n" + "=" * 80)
        logger.info("✅ PROCESSO FINALIZADO COM SUCESSO!")
        logger.info(f"   Tempo de execução: {tempo_duracao}")
        logger.info("=" * 80)
            
except Exception as e:
    logger.error(f"\n❌ ERRO CRÍTICO: {e}")
    import traceback
    logger.error(traceback.format_exc())

c:\Users\rodrigo.bernandes\AppData\Local\Programs\Python\Python312\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


# Atualizando o arquivo Excel Controle HC e Atestados
- Caminho: X:\Gestao_de_Pessoas\Analytics\10 - Relatórios\10.4 - HC e Atestados Médicos

In [9]:
# Caminho do arquivo
path_excel = r"X:\Gestao_de_Pessoas\Analytics\10 - Relatórios\10.4 - HC e Atestados Médicos\Controle_HC e Atestados.xlsb"
os.startfile(path_excel) # Abre o arquivo pelo Windows
time.sleep(60)
pyautogui.press('esc')
time.sleep(15)

# Utiliza o comando "Ir para" (Ctrl + G) para navegar até a aba e célula
pyautogui.hotkey('ctrl', 'g')
time.sleep(1)

# Digita o endereço completo
pyautogui.write('ATESTADOS!B5')
time.sleep(1)
pyautogui.press('enter')
time.sleep(3)

#Atualizando o arquivo
pyautogui.hotkey('alt', 'F5')
time.sleep(2)

print('----------------------------------------------------------------------------------------------------')
print('')
print('   ✅ Planilha atualizada com sucesso')
print('')
print('----------------------------------------------------------------------------------------------------')

----------------------------------------------------------------------------------------------------

   ✅ Planilha atualizada com sucesso

----------------------------------------------------------------------------------------------------


# Resumo de Finalização do Processo

In [10]:
tempo_1 = [id, datetime.today(), 8]

print('----------------------------------------------------------------------------------------------------')
print('')
print('     ✅  Processo finalizado')
print('')
print('     ⏱️   Tempo de execução:')
print('')
print(f'   {tempo_1[1] - tempo_0[1]}')
print('')
print('----------------------------------------------------------------------------------------------------')

----------------------------------------------------------------------------------------------------

     ✅  Processo finalizado

     ⏱️   Tempo de execução:

   0:09:31.730139

----------------------------------------------------------------------------------------------------
